# Corrected Sample-Size Stability Analysis

The original stability experiment used only 5 resamples per sample size, and
reported the MEAN accuracy at each size. Since a random sample's mean is an
unbiased estimator of the full-set accuracy regardless of sample size, this
mean is expected to stay roughly flat across n -- that finding was guaranteed
by construction, not evidence of stability.

What actually indicates stability is the SPREAD (variance / standard
deviation) of accuracy across many different random draws at each sample
size, which decreases as n increases if the benchmark is well-behaved. This
notebook recomputes stability properly: many more resamples (1000, not 5),
reporting the standard deviation and a 95% interval at each sample size,
rather than the mean.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import json
import random
import numpy as np
from pathlib import Path

RESULTS_DIRS = {
    'masked': Path("/content/drive/MyDrive/Thesis/results/masked"),
    'autoregressive': Path("/content/drive/MyDrive/Thesis/results/autoregressive"),
}
OUTPUT_DIR = Path("/content/drive/MyDrive/Thesis/results/stability_corrected")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_SIZES = [25, 50, 75, 100]
N_RESAMPLES = 1000   # up from 5
RANDOM_SEED = 42

print("Config set")

Config set


## Load per-pair correctness from existing results files\n\nReuses the already-computed `correct` field for every pair -- no re-scoring needed, this only re-does the resampling/statistics step.

In [4]:
def load_pair_correctness(results_dir):
    """Returns {model_name: {phenomenon: [list of booleans]}}"""
    all_model_correctness = {}
    for results_file in sorted(results_dir.glob("*_results.json")):
        with open(results_file, encoding='utf-8') as f:
            data = json.load(f)
        model_name = data.get('model', results_file.stem)
        per_phenomenon = data.get('per_phenomenon', {})

        model_correctness = {}
        for phenomenon, phen_data in per_phenomenon.items():
            correctness = [pair['correct'] for pair in phen_data['pairs']]
            model_correctness[phenomenon] = correctness
        all_model_correctness[model_name] = model_correctness

    return all_model_correctness


all_correctness = {}
for source, results_dir in RESULTS_DIRS.items():
    if results_dir.exists():
        loaded = load_pair_correctness(results_dir)
        all_correctness.update(loaded)
        print(f"Loaded {len(loaded)} models from {source}")

print(f"\nTotal models loaded: {len(all_correctness)}")

Loaded 3 models from masked
Loaded 8 models from autoregressive

Total models loaded: 11


## Corrected resampling: many more repetitions, reporting SPREAD not just mean

In [5]:
def corrected_stability(pair_correctness, sample_sizes=SAMPLE_SIZES, n_resamples=N_RESAMPLES, seed=RANDOM_SEED):
    """For each sample size, draw n_resamples random subsamples (with a
    different seed each time) and report mean, std, and a 95% interval
    (2.5th-97.5th percentile) of the resulting accuracy distribution."""
    results = {}
    rng = random.Random(seed)

    for size in sample_sizes:
        if size > len(pair_correctness):
            continue
        accs = []
        for _ in range(n_resamples):
            sample = rng.sample(pair_correctness, size)
            accs.append(sum(sample) / size)

        accs_arr = np.array(accs)
        results[size] = {
            'mean_accuracy': float(np.mean(accs_arr)),
            'std_accuracy': float(np.std(accs_arr)),
            'ci_2.5pct': float(np.percentile(accs_arr, 2.5)),
            'ci_97.5pct': float(np.percentile(accs_arr, 97.5)),
            'n_resamples': n_resamples,
        }
    return results

## Run for every model and phenomenon

In [6]:
corrected_results = {}

for model_name, phenomena in all_correctness.items():
    print(f"\n{model_name}")
    corrected_results[model_name] = {}
    for phenomenon, correctness in phenomena.items():
        stability = corrected_stability(correctness)
        corrected_results[model_name][phenomenon] = stability
        # Print just the std at n=25 and n=100 as a quick sanity check
        if 25 in stability and 100 in stability:
            print(f"  {phenomenon}: std at n=25 = {stability[25]['std_accuracy']:.3f}, "
                  f"std at n=100 = {stability[100]['std_accuracy']:.3f}")


FacebookAI/xlm-roberta-base
  noun_adjective_agreement: std at n=25 = 0.061, std at n=100 = 0.000
  aspect: std at n=25 = 0.076, std at n=100 = 0.000
  einai_agreement: std at n=25 = 0.057, std at n=100 = 0.000
  subject_verb_agreement: std at n=25 = 0.045, std at n=100 = 0.000
  negations: std at n=25 = 0.052, std at n=100 = 0.000
  case_selection: std at n=25 = 0.053, std at n=100 = 0.000

google-bert/bert-base-multilingual-cased
  noun_adjective_agreement: std at n=25 = 0.051, std at n=100 = 0.000
  aspect: std at n=25 = 0.081, std at n=100 = 0.000
  einai_agreement: std at n=25 = 0.065, std at n=100 = 0.000
  subject_verb_agreement: std at n=25 = 0.030, std at n=100 = 0.000
  negations: std at n=25 = 0.066, std at n=100 = 0.000
  case_selection: std at n=25 = 0.073, std at n=100 = 0.000

nlpaueb/bert-base-greek-uncased-v1
  noun_adjective_agreement: std at n=25 = 0.057, std at n=100 = 0.000
  aspect: std at n=25 = 0.056, std at n=100 = 0.000
  einai_agreement: std at n=25 = 0.040,

## Save

In [7]:
output_path = OUTPUT_DIR / "corrected_stability_all_models.json"
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(corrected_results, f, ensure_ascii=False, indent=2)

print(f"Saved to {output_path}")

Saved to /content/drive/MyDrive/Thesis/results/stability_corrected/corrected_stability_all_models.json


## Summary: average std (spread) at each sample size, across all models/phenomena\n\nThis is the number that actually tells you whether stability improves with n -- std should DECREASE as sample size increases, if the benchmark behaves as expected.

In [8]:
import pandas as pd

rows = []
for model_name, phenomena in corrected_results.items():
    for phenomenon, stability in phenomena.items():
        for size, stats in stability.items():
            rows.append({
                'model': model_name,
                'phenomenon': phenomenon,
                'n': size,
                'std': stats['std_accuracy'],
                'ci_width': stats['ci_97.5pct'] - stats['ci_2.5pct'],
            })

df = pd.DataFrame(rows)
summary = df.groupby('n')[['std', 'ci_width']].mean()
print("Average standard deviation and 95% CI width, by sample size (across all models/phenomena):\n")
print(summary.round(4))

Average standard deviation and 95% CI width, by sample size (across all models/phenomena):

        std  ci_width
n                    
25   0.0581    0.2188
50   0.0334    0.1279
75   0.0193    0.0719
100  0.0000    0.0000
